In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from building_models.commons_functions.parsers_commons import ParsersCommons
from building_models.utils.constants import COLUMNS_TO_WORK
from building_models.utils.utils_functions import UtilsFunctions
import pandas as pd

In [3]:
path_export = "../../processed_dataset/"
path_input = "../../raw_dataset/"
metadata_file = "../../raw_dataset/raw_data_description.xlsx"
name_task = "antioxidant_classification"
name_source = "AMPDB v1"

- Read doc

In [4]:
df = pd.read_csv(f"{path_input}/{name_source}/Antioxidant dataset.tsv", sep='\t')
df_data = df[[' Sequence']]
df_data = df_data.rename(columns={' Sequence': 'sequence'})
df_data

,sequence
0,MSALGAVIALLLWGQLFAVDSGNDVTDIADDGCPKPPEIAHGYVEH...
1,MTCKMSQLERNIETIINTFHQYSVKLGHPDTLNQGEFKELVRKDLQ...
2,MRALGAVVTLLLWGQLFAVELGNDATDIEDDSCPKPPEIANGYVEH...
3,EDTGSEATNNTEVSLPKPPVIENGYVEHMIRYQCKPFYKLHTEGDG...
4,MPSELEKALSNLIDVYHNYSNIQGNHHALYKNDFKKMVTTECPQFV...
...,...
351,MSALQAVVALLLCGQLFAVQTTETTTATDDSCLKPPEIANGYLEHL...
352,MSALGAVIALLLWGQLFAVDSGNDVTDIADDSCPKPPEIANGYVEH...
353,CAADTGSEATDHAEVSVPKPPEIENGYVQHLIRYQCKPLYRLRTEG...
354,CSVLPAVITLLLCGQLLAVETGSEAAAGSCPKAPEIANGHVEYSVR...


In [5]:
df_data['label'] = 1

- Checking duplicates

In [6]:
df_consistent_duplicates, df_errors, df_unique = ParsersCommons.processing_duplicated(
    df_data, group_seq= "sequence",
    label_col= "label")
df_consistent_duplicates.shape, df_errors.shape, df_unique.shape

((10, 3), (0, 0), (334, 2))

In [7]:
data_correct = pd.concat([df_consistent_duplicates, df_unique], axis=0, ignore_index=True)
data_correct = data_correct.drop(columns=["n_duplicates"])
data_correct.head()

,sequence,label
0,KPAEIEHGYVEHLIKYRCNPYYQLRGSGDGTYKCDEDHMWVSSEAG...,1
1,KPAEIEHGYVEHLIKYRCNPYYQLRGSGDGTYKCDEDHMWVSSEAG...,1
2,MRALGAVIALLLWGQLFAEDTGSEATNNTEVSLPKPPVIENGYVEH...,1
3,MRALGAVVALLFWGQIFAVDTGNATDNTEVSLPKPPEIENGYAEHF...,1
4,MRALGAVVALLLCGQLSAADTGSEATDHGEVSVPKPPEIENGYVEH...,1


- Reading metadata

In [8]:
metadata_file = ParsersCommons.read_metadata(metadata_file, name_source=name_source, columns_to_select=COLUMNS_TO_WORK)
metadata_file.head()

,name dataset,name source,type source,static-dynamic,license,reports constant updates,year of publication,last update date,download date,file format,protein format,category dataset,task,obtaining negative dataset,obtaining positive dataset,repository or server,publication,unit of measurement
2,Antioxidant dataset.tsv,AMPDB v1,Database,Static,No information,Yes,2023,2023-04-28,2025-08-01,tsv,Sequence,antioxidant,Antioxidant,No information,No information,https://bblserver.org.in/ampdb/,https://www.nature.com/articles/s41598-023-450...,No information


In [9]:
dict_metadata = ParsersCommons.create_metadata_from_file(metadata_file)
dict_metadata

{'name dataset': 'Antioxidant dataset.tsv',
 'name source': 'AMPDB v1',
 'type source': 'Database',
 'static-dynamic': 'Static',
 'license': 'No information',
 'reports constant updates': 'Yes',
 'year of publication': 2023,
 'last update date': Timestamp('2023-04-28 00:00:00'),
 'download date': Timestamp('2025-08-01 00:00:00'),
 'file format': 'tsv',
 'protein format': 'Sequence',
 'category dataset': 'antioxidant',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'No information',
 'obtaining positive dataset': 'No information',
 'repository or server': 'https://bblserver.org.in/ampdb/',
 'publication': 'https://www.nature.com/articles/s41598-023-45016-3',
 'unit of measurement': 'No information',
 'number_of_sources': 1,
 'processing_date': '2026-04-16 18:26:38'}

In [10]:
dict_metadata['number_of_records'] = df_data.shape[0]
dict_metadata['number_of_collected_sequences'] = df_data.shape[0]
dict_metadata['number_of_unique_sequences'] = data_correct.shape[0]
dict_metadata['positive_examples'] = data_correct[data_correct["label"] == 1].shape[0]
dict_metadata['negative_examples'] = data_correct[data_correct["label"] == 0].shape[0]
dict_metadata['number_of_sequences_with_errors'] = df_errors.shape[0]
dict_metadata

{'name dataset': 'Antioxidant dataset.tsv',
 'name source': 'AMPDB v1',
 'type source': 'Database',
 'static-dynamic': 'Static',
 'license': 'No information',
 'reports constant updates': 'Yes',
 'year of publication': 2023,
 'last update date': Timestamp('2023-04-28 00:00:00'),
 'download date': Timestamp('2025-08-01 00:00:00'),
 'file format': 'tsv',
 'protein format': 'Sequence',
 'category dataset': 'antioxidant',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'No information',
 'obtaining positive dataset': 'No information',
 'repository or server': 'https://bblserver.org.in/ampdb/',
 'publication': 'https://www.nature.com/articles/s41598-023-45016-3',
 'unit of measurement': 'No information',
 'number_of_sources': 1,
 'processing_date': '2026-04-16 18:26:38',
 'number_of_records': 356,
 'number_of_collected_sequences': 356,
 'number_of_unique_sequences': 344,
 'positive_examples': 344,
 'negative_examples': 0,
 'number_of_sequences_with_errors': 0}

- Export data

In [11]:
UtilsFunctions.make_directory(f"{path_export}{name_task}/{name_source}")
UtilsFunctions.export_json(f"{path_export}{name_task}/{name_source}/metadata_{name_source}.json", dict_metadata)
data_correct.to_csv(f"{path_export}{name_task}/{name_source}/processed_data.csv", index=False)